# Catalog Helpers Notebook

This notebook provides reusable utilities for schema and table management in the CDC pipeline.

## Functions
- `ensure_schema_exists(catalog, schema)` - Create schema if not exists
- `get_existing_schema(spark, table_fqn)` - Get existing Delta table schema
- `get_existing_columns(spark, table_fqn)` - Get set of column names
- `create_silver_table_rental(spark, table_fqn)` - Create rental Silver table
- `create_silver_table_film(spark, table_fqn)` - Create film Silver table
- `create_silver_table_payment(spark, table_fqn)` - Create payment Silver table
- `build_merge_clauses(existing_columns, core_fields, ...)` - Build MERGE update/insert clauses
- `execute_merge(spark, delta_table, df, update_set, insert_values, ...)` - Execute MERGE

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, coalesce, lit
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType, DecimalType


def ensure_schema_exists(catalog: str, schema: str) -> None:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
    print(f"Schema {catalog}.{schema} is ready")


def get_existing_schema(spark, table_fqn: str) -> StructType | None:
    try:
        return spark.table(table_fqn).schema
    except Exception:
        return None


def get_existing_columns(spark, table_fqn: str) -> set:
    try:
        return set(spark.table(table_fqn).columns)
    except Exception:
        return set()


def create_silver_table_rental(spark, table_fqn: str) -> None:
    """Create the Silver rental table for dvdrental CDC upserts."""
    spark.sql(
        f"""
        CREATE TABLE IF NOT EXISTS {table_fqn} (
            rental_id       INT         NOT NULL,
            rental_date     TIMESTAMP,
            inventory_id    INT,
            customer_id     INT,
            return_date     TIMESTAMP,
            staff_id        INT,
            last_update     TIMESTAMP,
            last_inserted_dt TIMESTAMP,
            last_updated_dt  TIMESTAMP  NOT NULL
        ) USING DELTA
        """
    )
    print(f"Silver rental table {table_fqn} is ready")


def create_silver_table_film(spark, table_fqn: str) -> None:
    """Create the Silver film table for dvdrental CDC upserts."""
    spark.sql(
        f"""
        CREATE TABLE IF NOT EXISTS {table_fqn} (
            film_id          INT            NOT NULL,
            title            STRING         NOT NULL,
            description      STRING,
            release_year     INT,
            language_id      INT,
            rental_duration  INT,
            rental_rate      DECIMAL(4,2),
            length           INT,
            replacement_cost DECIMAL(5,2),
            rating           STRING,
            last_update      TIMESTAMP,
            last_inserted_dt TIMESTAMP,
            last_updated_dt  TIMESTAMP      NOT NULL
        ) USING DELTA
        """
    )
    print(f"Silver film table {table_fqn} is ready")


def create_silver_table_payment(spark, table_fqn: str) -> None:
    """Create the Silver payment table for dvdrental CDC upserts."""
    spark.sql(
        f"""
        CREATE TABLE IF NOT EXISTS {table_fqn} (
            payment_id      INT            NOT NULL,
            customer_id     INT,
            staff_id        INT,
            rental_id       INT,
            amount          DECIMAL(5,2),
            payment_date    TIMESTAMP,
            last_inserted_dt TIMESTAMP,
            last_updated_dt  TIMESTAMP     NOT NULL
        ) USING DELTA
        """
    )
    print(f"Silver payment table {table_fqn} is ready")


def build_merge_clauses(
    existing_columns: set,
    core_fields: list,
    include_inserted_dt: bool = True,
    include_updated_dt: bool = True
) -> tuple[dict, dict]:
    """
    Build dynamic MERGE update and insert clauses based on existing schema.
    The first element of core_fields is treated as the primary key (excluded from update_set).
    """
    pk = core_fields[0] if core_fields else None
    update_set = {}
    insert_values = {}

    for field in core_fields:
        if field in existing_columns:
            if field != pk:
                update_set[field] = f"s.{field}"
            insert_values[field] = f"s.{field}"

    if include_inserted_dt and "last_inserted_dt" in existing_columns:
        update_set["last_inserted_dt"] = "COALESCE(s.event_time, t.last_inserted_dt)"
        insert_values["last_inserted_dt"] = "s.event_time"

    if include_updated_dt and "last_updated_dt" in existing_columns:
        update_set["last_updated_dt"] = "s.event_time"
        insert_values["last_updated_dt"] = "s.event_time"

    return update_set, insert_values


def execute_merge(
    spark,
    delta_table: DeltaTable,
    latest_df,
    update_set: dict,
    insert_values: dict,
    join_condition: str = "t.id = s.id",
    delete_condition: str = "s.op = 'd'",
    update_condition: str = "s.op <> 'd'"
) -> None:
    """Execute a MERGE operation with dynamic clauses."""
    merge_builder = (
        delta_table.alias("t")
        .merge(latest_df.alias("s"), join_condition)
    )

    if update_set:
        merge_builder = merge_builder.whenMatchedUpdate(
            condition=update_condition,
            set=update_set
        )

    merge_builder = merge_builder.whenMatchedDelete(condition=delete_condition)

    if insert_values:
        merge_builder = merge_builder.whenNotMatchedInsert(
            condition=update_condition,
            values=insert_values
        )

    merge_builder.execute()

In [ ]:
# ---------------------------------------------------------------------------
# Monitoring schema + DQ / PII / Key-management DDL
# Call ensure_monitoring_tables(spark, catalog) once per cluster start.
# ---------------------------------------------------------------------------

def ensure_monitoring_tables(spark, catalog: str = "workspace") -> None:
    """Create all monitoring-layer tables if they do not exist."""

    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.monitoring")

    # ------------------------------------------------------------------
    # Task 1.1 — PII column registry
    # ------------------------------------------------------------------
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {catalog}.monitoring.pii_column_registry (
            table_name     STRING,
            column_name    STRING,
            sensitivity    STRING,
            subject_id_col STRING,
            encrypt        BOOLEAN,
            tagged_at      TIMESTAMP
        ) USING DELTA
    """)
    print(f"{catalog}.monitoring.pii_column_registry — ready")

    # ------------------------------------------------------------------
    # Task 1.2 — Unified DQ results
    # ------------------------------------------------------------------
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {catalog}.monitoring.dq_results (
            run_id         STRING,
            layer          STRING,
            table_name     STRING,
            check_name     STRING,
            status         STRING,
            observed_value DOUBLE,
            threshold      DOUBLE,
            message        STRING,
            checked_at     TIMESTAMP
        ) USING DELTA
        TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
    """)
    print(f"{catalog}.monitoring.dq_results — ready")

    # ------------------------------------------------------------------
    # Task 1.3 — Subject key store (crypto-shredding)
    # ------------------------------------------------------------------
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {catalog}.monitoring.subject_key_store (
            subject_id    STRING,
            subject_type  STRING,
            encrypted_dek BINARY,
            kek_version   STRING,
            created_at    TIMESTAMP,
            shredded_at   TIMESTAMP
        ) USING DELTA
    """)
    print(f"{catalog}.monitoring.subject_key_store — ready")

    # ------------------------------------------------------------------
    # Task 2.5 — Erasure management tables
    # ------------------------------------------------------------------
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {catalog}.monitoring.erasure_requests (
            request_id   STRING,
            subject_id   STRING,
            subject_type STRING,
            requested_at TIMESTAMP,
            status       STRING,
            requester    STRING,
            notes        STRING
        ) USING DELTA
    """)
    print(f"{catalog}.monitoring.erasure_requests — ready")

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {catalog}.monitoring.erasure_registry (
            subject_id    STRING,
            subject_type  STRING,
            suppressed_at TIMESTAMP
        ) USING DELTA
    """)
    print(f"{catalog}.monitoring.erasure_registry — ready")

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {catalog}.monitoring.erasure_audit_log (
            request_id   STRING,
            step         STRING,
            completed_at TIMESTAMP,
            evidence     STRING,
            operator     STRING
        ) USING DELTA
    """)
    print(f"{catalog}.monitoring.erasure_audit_log — ready")

    print("All monitoring tables ready.")

In [ ]:
# ---------------------------------------------------------------------------
# Task 1.2 — write_dq_result helper
# Callable from any notebook via %run ./helpers/NB_catalog_helpers
# ---------------------------------------------------------------------------
from datetime import datetime, timezone
from pyspark.sql import Row


def write_dq_result(
    spark,
    run_id: str,
    layer: str,
    table_name: str,
    check_name: str,
    status: str,
    observed_value: float = None,
    threshold: float = None,
    message: str = None,
    catalog: str = "workspace",
) -> None:
    """
    Append one DQ result row to monitoring.dq_results.

    Parameters
    ----------
    run_id         : unique identifier for this pipeline run (e.g. UUID or batch_id)
    layer          : 'bronze' | 'silver' | 'vault' | 'gold'
    table_name     : fully-qualified table name (e.g. 'silver.silver_rental')
    check_name     : short identifier for the check (e.g. 'pk_uniqueness')
    status         : 'PASS' | 'FAIL' | 'WARN'
    observed_value : numeric result of the check (e.g. duplicate count, null rate)
    threshold      : threshold used for PASS/FAIL decision
    message        : human-readable description
    catalog        : Unity Catalog name (default 'workspace')
    """
    row = Row(
        run_id         = run_id,
        layer          = layer,
        table_name     = table_name,
        check_name     = check_name,
        status         = status,
        observed_value = float(observed_value) if observed_value is not None else None,
        threshold      = float(threshold)      if threshold      is not None else None,
        message        = message,
        checked_at     = datetime.now(timezone.utc),
    )
    spark.createDataFrame([row]).write.format("delta").mode("append").saveAsTable(
        f"{catalog}.monitoring.dq_results"
    )

In [ ]:
# ---------------------------------------------------------------------------
# Task 2.1 — Bronze quarantine table DDL
# Task 2.2 — Bronze TTL helper
# ---------------------------------------------------------------------------

def ensure_bronze_quarantine(spark, catalog: str = "workspace") -> None:
    """Create bronze.quarantine table if it does not exist."""
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {catalog}.bronze.quarantine (
            topic             STRING,
            partition         INT,
            offset            LONG,
            kafka_timestamp   TIMESTAMP,
            message_key       STRING,
            value             STRING,
            ingested_at       TIMESTAMP,
            source_schema     STRING,
            table_name        STRING,
            quarantine_reason STRING,
            quarantined_at    TIMESTAMP
        ) USING DELTA
    """)
    print(f"{catalog}.bronze.quarantine — ready")


def set_bronze_ttl(spark, table_fqn: str, retain_days: int = 30) -> None:
    """
    Apply 30-day file and log retention to a Bronze Delta table (GDPR TTL).
    Idempotent — safe to call on every batch write.
    """
    hours = retain_days * 24
    spark.sql(f"""
        ALTER TABLE {table_fqn}
        SET TBLPROPERTIES (
            'delta.deletedFileRetentionDuration' = 'interval {hours} hours',
            'delta.logRetentionDuration'         = 'interval {hours} hours'
        )
    """)

In [ ]:
# ---------------------------------------------------------------------------
# Task 3.1 — Vault load log DDL
# ---------------------------------------------------------------------------

def ensure_vault_load_log(spark, catalog: str = "workspace") -> None:
    """Create monitoring.vault_load_log if it does not exist."""
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {catalog}.monitoring.vault_load_log (
            run_id       STRING,
            layer        STRING,
            table_name   STRING,
            rows_loaded  LONG,
            check_name   STRING,
            status       STRING,
            message      STRING,
            loaded_at    TIMESTAMP
        ) USING DELTA
    """)
    print(f"{catalog}.monitoring.vault_load_log — ready")

In [ ]:
# ---------------------------------------------------------------------------
# Task 4.4 — Row count / volume anomaly detection
# ---------------------------------------------------------------------------

def check_volume_anomaly(
    spark,
    run_id: str,
    table_name: str,
    current_count: int,
    lookback_days: int = 7,
    threshold_pct: float = 0.5,
    catalog: str = "workspace",
) -> str:
    """
    Compare current_count against a rolling {lookback_days}-day average from dq_results.

    Status:
    - PASS  : current_count >= average * threshold_pct
    - WARN  : current_count < average * threshold_pct (but > 0)
    - FAIL  : current_count == 0

    Results are written to monitoring.dq_results.

    Parameters
    ----------
    spark          : SparkSession
    run_id         : pipeline run identifier
    table_name     : short or qualified table name (used as filter key)
    current_count  : row count in current batch / micro-batch
    lookback_days  : rolling window for average (default 7)
    threshold_pct  : fraction of average below which WARN fires (default 0.5 = 50%)
    catalog        : Unity Catalog name
    """
    if current_count == 0:
        status = "FAIL"
        message = f"ZERO rows in batch for {table_name}"
    else:
        # Rolling average from dq_results for this table's volume checks
        avg_row = spark.sql(f"""
            SELECT AVG(observed_value) AS avg_count
            FROM {catalog}.monitoring.dq_results
            WHERE table_name = '{table_name}'
              AND check_name  = 'batch_row_count'
              AND status      IN ('PASS', 'WARN')
              AND checked_at  >= current_timestamp() - INTERVAL {lookback_days} DAYS
        """).collect()[0]
        avg_count = avg_row["avg_count"]

        if avg_count is None or avg_count == 0:
            # No history yet — treat as PASS
            status  = "PASS"
            message = None
        elif current_count < avg_count * threshold_pct:
            status  = "WARN"
            message = (
                f"Volume drop: current={current_count}, "
                f"{lookback_days}d_avg={avg_count:.0f}, "
                f"threshold={threshold_pct*100:.0f}%"
            )
        else:
            status  = "PASS"
            message = None

    write_dq_result(
        spark,
        run_id         = run_id,
        layer          = "bronze",
        table_name     = table_name,
        check_name     = "batch_row_count",
        status         = status,
        observed_value = float(current_count),
        threshold      = threshold_pct,
        message        = message,
        catalog        = catalog,
    )
    return status

## Usage Example

```python
%run ./helpers/NB_catalog_helpers

ensure_schema_exists(CONFIG['catalog'], CONFIG['silver_schema'])
create_silver_table_rental(spark, silver_table_fqn)

existing_columns = get_existing_columns(spark, silver_table_fqn)
update_set, insert_values = build_merge_clauses(existing_columns, core_fields)
execute_merge(spark, delta_table, latest, update_set, insert_values, join_condition='t.rental_id = s.rental_id')
```